[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/36_int8_quantization.ipynb)

# 🔴 Hard: INT8 Quantized Linear

Implement a **post-training quantized linear layer** using INT8 weights.

### Signature
```python
class Int8Linear(nn.Module):
    def __init__(self, weight: Tensor, bias: Tensor = None): ...
    def forward(self, x: Tensor) -> Tensor: ...
```

### Quantization (per-channel)
1. `scale = weight.abs().max(dim=1) / 127`
2. `weight_int8 = round(weight / scale).clamp(-128, 127).to(int8)`
3. Store as `register_buffer` (not trainable)
4. Forward: dequantize (`int8.float() * scale`) then matmul

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 4.0 MB/s eta 0:00:00


In [3]:
import torch
import torch.nn as nn

In [11]:
# ✏️ YOUR IMPLEMENTATION HERE

class Int8Linear(nn.Module):
    def __init__(self, weight, bias=None):
        super().__init__()

        max_val=weight.abs().max(dim=1,keepdim=True)[0]

        scale=max_val/127.0
        self.register_buffer('scale',scale)

        weight_int8=torch.round(weight/self.scale).clamp(-128,127).to(torch.int8)
        self.register_buffer('weight_int8',weight_int8)

        if bias is not None:
          self.register_buffer('bias', bias)
        else:
            self.bias = None


    def forward(self, x):
      dequant_weight=self.weight_int8.to(x.dtype)*self.scale

      output=torch.matmul(x,dequant_weight.t())
      if self.bias is not None:
            output += self.bias

      return output




In [12]:
# 🧪 Debug
w = torch.randn(8, 4)
q = Int8Linear(w)
x = torch.randn(2, 4)
print('Output:', q(x).shape)
print('dtype:', q.weight_int8.dtype)
print('Max quant error:', (w - q.weight_int8.float() * q.scale).abs().max().item())

Output: torch.Size([2, 8])
dtype: torch.int8
Max quant error: 0.006759822368621826


In [13]:
# ✅ SUBMIT
from torch_judge import check
check('int8_quantization')


🧪 Testing: INT8 Quantized Linear (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Weight is int8 (0.6ms)
  ✅ [2/5] Values in [-128, 127] (0.4ms)
  ✅ [3/5] Dequantized close to original (1.6ms)
  ✅ [4/5] Forward output shape (0.5ms)
  ✅ [5/5] Weight is buffer not parameter (0.4ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (3.5ms total)
  Progress saved. Run status() to see your dashboard.

